# Stage 1 Thesis Statistics Analysis

**Mục tiêu:** Phân tích 417 videos (70 categories) để chọn candidate window sizes cho Stage 2.

**Input:** `analysis/stage1/default/stage1_consolidated.parquet` (114 MB)

**Output:** 10 figures, 6 tables, candidate_window_sizes.json

## Section 0: Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style
sns.set_palette("colorblind")
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

# Load parquet
df = pd.read_parquet("analysis/stage1/default/stage1_consolidated.parquet")

# Type coercion (CSV string → numeric)
num_cols = ["frame_idx", "num_frames_total",
            "maskmem_max_distance", "maskmem_min_distance", "maskmem_mean_distance",
            "n_maskmem_selected", "scan_depth", "n_candidates_rejected",
            "min_iou_of_selected", "mean_iou_of_selected",
            "prev_predicted_iou", "inference_time_ms",
            "membank_ram_bytes", "process_rss_bytes", "gpu_vram_bytes"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop frame 0 sentinel (max_distance = -1, no memory bank)
df_valid = df[df["maskmem_max_distance"] >= 0].copy()

# Explode Distribution A (per-selection distance)
df_A = df_valid[["category", "video_name", "frame_idx", "maskmem_distances"]].copy()
df_A["distance"] = df_A["maskmem_distances"].apply(lambda s: json.loads(s) if s else [])
df_A = df_A.explode("distance").dropna(subset=["distance"])
df_A["distance"] = df_A["distance"].astype(int)

print(f"✓ Loaded {len(df):,} rows")
print(f"✓ Videos: {df['video_name'].nunique()}")
print(f"✓ Categories: {df['category'].nunique()}")
print(f"✓ Valid frames (after drop frame 0): {len(df_valid):,}")
print(f"✓ Distribution A (selections): {len(df_A):,}")

✓ Loaded 1,063,842 rows
✓ Videos: 415
✓ Categories: 70
✓ Valid frames (after drop frame 0): 1,063,012
✓ Distribution A (selections): 6,369,659


In [2]:
def save_fig(fig, name: str) -> None:
    """Save figure as PNG (300 DPI) + PDF (vector)"""
    Path("figures/stage1").mkdir(parents=True, exist_ok=True)
    fig.savefig(f"figures/stage1/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"figures/stage1/{name}.pdf", bbox_inches="tight")
    print(f"✓ Saved figures/stage1/{name}.{{png,pdf}}")

def save_table(df_table: pd.DataFrame, name: str, caption: str = "") -> None:
    """Save table as CSV + Markdown + LaTeX"""
    Path("tables/stage1").mkdir(parents=True, exist_ok=True)
    base = f"tables/stage1/{name}"
    df_table.to_csv(f"{base}.csv", index=False)
    with open(f"{base}.md", "w") as f:
        f.write(f"**{caption}**\n\n")
        f.write(df_table.to_markdown(index=False))
    with open(f"{base}.tex", "w") as f:
        f.write(f"% {caption}\n")
        f.write(df_table.to_latex(index=False, escape=False))
    print(f"✓ Saved tables/stage1/{name}.{{csv,md,tex}}")

print("✓ Helper functions defined")

✓ Helper functions defined


In [3]:
# Sanity checks
assert df["video_name"].nunique() >= 410, "Expected ≥410 videos"
assert df["category"].nunique() == 70, "Expected 70 categories"
assert df_valid["maskmem_max_distance"].min() >= 1, "Max distance should be ≥1"
assert df_A["distance"].min() >= 1, "Selection distance should be ≥1"
print(f"✓ All sanity checks passed ({df['video_name'].nunique()} videos, {df['category'].nunique()} categories)")

✓ All sanity checks passed (415 videos, 70 categories)


## Section 1: Dataset Overview & Coverage

In [4]:
# Compute overview statistics
overview = pd.DataFrame([
    ("Videos analyzed", f"{df['video_name'].nunique()} / 420"),
    ("Categories covered", f"{df['category'].nunique()} / 70"),
    ("Total frames (incl. frame 0)", f"{len(df):,}"),
    ("Valid frames (drop frame 0)", f"{len(df_valid):,}"),
    ("Total selections (Dist A)", f"{len(df_A):,}"),
    ("Mean frames/video", f"{df.groupby('video_name').size().mean():.0f}"),
    ("Median frames/video", f"{df.groupby('video_name').size().median():.0f}"),
    ("Min frames/video", f"{df.groupby('video_name').size().min()}"),
    ("Max frames/video", f"{df.groupby('video_name').size().max()}"),
], columns=["Metric", "Value"])

print(overview.to_markdown(index=False))

| Metric                       | Value     |
|:-----------------------------|:----------|
| Videos analyzed              | 415 / 420 |
| Categories covered           | 70 / 70   |
| Total frames (incl. frame 0) | 1,063,842 |
| Valid frames (drop frame 0)  | 1,063,012 |
| Total selections (Dist A)    | 6,369,659 |
| Mean frames/video            | 2563      |
| Median frames/video          | 2160      |
| Min frames/video             | 1000      |
| Max frames/video             | 9193      |


In [5]:
save_table(overview, "01_stage1_overview",
           caption="Table 1.1: Stage 1 Dataset Overview")

✓ Saved tables/stage1/01_stage1_overview.{csv,md,tex}


In [6]:
# Compute overview statistics
overview = pd.DataFrame([
    ("Videos analyzed", f"{df['video_name'].nunique()} / 420"),
    ("Categories covered", f"{df['category'].nunique()} / 70"),
    ("Total frames (incl. frame 0)", f"{len(df):,}"),
    ("Valid frames (drop frame 0)", f"{len(df_valid):,}"),
    ("Total selections (Dist A)", f"{len(df_A):,}"),
    ("Mean frames/video", f"{df.groupby('video_name').size().mean():.0f}"),
    ("Median frames/video", f"{df.groupby('video_name').size().median():.0f}"),
    ("Min frames/video", f"{df.groupby('video_name').size().min()}"),
    ("Max frames/video", f"{df.groupby('video_name').size().max()}"),
], columns=["Metric", "Value"])

print(overview.to_markdown(index=False))

| Metric                       | Value     |
|:-----------------------------|:----------|
| Videos analyzed              | 415 / 420 |
| Categories covered           | 70 / 70   |
| Total frames (incl. frame 0) | 1,063,842 |
| Valid frames (drop frame 0)  | 1,063,012 |
| Total selections (Dist A)    | 6,369,659 |
| Mean frames/video            | 2563      |
| Median frames/video          | 2160      |
| Min frames/video             | 1000      |
| Max frames/video             | 9193      |


In [7]:
save_table(overview, "01_stage1_overview",
           caption="Table 1.1: Stage 1 Dataset Overview")

✓ Saved tables/stage1/01_stage1_overview.{csv,md,tex}


## Section 2: Distribution A & B Analysis

## Section 3: Coverage Curves

## Section 4: Per-Category Analysis

## Section 5: Per-Attribute Analysis

## Section 6: Memory Bank RAM Preliminary

## Section 7: Candidate Window Sizes Selection

## Section 8: Summary Tables for Thesis